### Recuit Simulé 
_______________________________________________________________________________________________________________
### Auteur : Luc Carlos Asso

### _______________________________________________________________________________________________________________

### Importation des Packages

### _______________________________________________________________________________________________________________

In [ ]:
import re
import numpy as np
import random
import matplotlib.pyplot as plt
import time 

### _______________________________________________________________________________________________________________

### Recuit en mode classe

### """  Informations :
            ‣ read_instance(filename) : recupération des instances
            ‣ Fonction_objectif( x : np.ndarray) : Fonction Objectif
            ‣ Solution_Initiale() : Solution Intiale
            ‣ voisin_swp(x : np.ndarray): Voisin de x
            ‣ Sauvegarder(solution : np.ndarray) : Sauvegarder les solutions
            ‣ Meilleurs_solutions() : les meilleurs solutions réalisables
     """

In [ ]:
class recuit_simule :
    
    @staticmethod
    def read_instance(filename):
        with open(filename, "r") as f:
            lines = [line.strip() for line in f if line.strip()]
        n = int(re.findall(r'\d+', lines[1])[0])                                      # -------- n --------------
        profits = [[0]*n for _ in range(n)]                                           # -------- profits --------
        idx = 3
        for i in range(n):
            nums = list(map(int, re.findall(r'-?\d+', lines[idx])))
            for j in range(i+1):
                profits[i][j] = nums[j]
                profits[j][i] = nums[j]
            idx += 1

        idx += 1  # saute "#Weights:"
        weights = list(map(int, re.findall(r'-?\d+', lines[idx])))                    # -------- poids -----------
        capacity = int(re.findall(r'\d+', lines[idx+1])[0])                           # -------- capacity --------
        k = int(re.findall(r'\d+', lines[idx+2])[0])                                  # -------- k ---cardinalité-
        return n, profits, weights, capacity, k

    
    n, P, W, C, k = recuit_simule.read_instance("Instance.txt")
    P = np.array(P)
    W = np.array(W)  
    
    @staticmethod
    def Fonction_objectif( x : np.ndarray) :
        Val1 = 0
        for i in range(recuit_simule.n) :
            Val1+=recuit_simule.P[i,i]*x[i]*x[i]
            for j in range(i+1,recuit_simule.n) : 
                    Val1+= recuit_simule.P[i,j]*x[i]*x[j]
        return Val1
        
 
    
    @staticmethod
    def Solution_Initiale() : 
        dict_test = {}
        dict_stock = {}
        nb_objet= recuit_simule.n
        W_T = sum(recuit_simule.W)
        Sol_I = [1]*recuit_simule.n
        profits = np.sum(recuit_simule.P, axis=1)
        for i in range(recuit_simule.n) : 
            dict_test[i] = profits[i]/recuit_simule.W[i]
        for i in range(recuit_simule.n - recuit_simule.k) : 
            ind = sorted(dict_test, key=dict_test.get)[0]
            Sol_I[ind]=0
            W_T-= recuit_simule.W[ind]
            nb_objet-=1
            dict_stock[ind]= dict_test[ind]
            del dict_test[ind]
            if W_T <= recuit_simule.C and nb_objet == recuit_simule.k :
                break
            while nb_objet == recuit_simule.k and W_T > recuit_simule.C : 
                ind = sorted(dict_test, key=dict_test.get)[0]
                j=0
                for i in list(dict_stock.keys())[::-1] :
                    if recuit_simule.W[ind] - recuit_simule.W[i] > 0 :
                        Sol_I[ind]=0
                        Sol_I[i]=1
                        W_T+= recuit_simule.W[i] - recuit_simule.W[ind]  
                        j-=1
                        dict_test[i]= profits[i]/recuit_simule.W[i]
                        del dict_test[ind]
                        break
                    j+=1
                if j == len(list(dict_stock.keys())) :
                    del dict_test[ind]
        return np.array(Sol_I)           
                               
    @staticmethod
    def voisin_swp(x: np.ndarray):
        v = x.copy()
        essais=0
        while  essais < 50 :
            indices_1 = [i for i in range(len(v)) if v[i] == 1]
            indices_0 = [i for i in range(len(v)) if v[i] == 0]
            i = random.choice(indices_1)
            j = random.choice(indices_0)
            v[i] = 0
            v[j] = 1
            if np.dot(v, recuit_simule.W) <= recuit_simule.C:
                break
            essais += 1
        return v
        
    """
        Paramètres 
            ‣ T  Température
            ‣ taux_d  taux de décroissance
            ‣ ɛ : paramètre de condition 
            ‣ λ : paramètre de la fonction objectif
            ‣ k : nombre de 1 dans la solution Initiale 
     """
   
    Historique_Sol=[]  # Historique des solutions
    @staticmethod 
    def Recuit_simulé(T : float ,taux_d : float, ɛ : float, R : int ):
        const_cond_pause = int( (((T- ɛ)/taux_d) + 1)/3 ) - 2
        pas_pause=0
        while T > ɛ :
            for i in  range(R) :
                Sol_initiale = recuit_simule.Historique_Sol[-1]
                V_s = recuit_simule.voisin_swp(Sol_initiale)
                F_V_s = recuit_simule.Fonction_objectif(V_s)         # image de  V_s par Fonction_objectif
                F_s = recuit_simule.Fonction_objectif(Sol_initiale)  # image de la solution initiale par Fonction_objectif
                if F_V_s >  F_s : 
                    Sol_initiale = V_s
                else:
                    r = random.random()
                    ΔF =  F_V_s -  F_s
                    if r <= np.exp(ΔF/T) :
                        Sol_initiale = V_s
                recuit_simule.Sauvegarder(Sol_initiale)                               
            T=T*taux_d
            pas_pause = pas_pause + 1
            if pas_pause == const_cond_pause :                     # pause 
                    time.sleep(60)
                    pas_pause = 0
                    

        return Sol_initiale
        

    @staticmethod
    
    def Sauvegarder(solution : np.ndarray) :                       # Sauvegarder les solutions
        recuit_simule.Historique_Sol.append(solution.copy())

    @staticmethod
    def Meilleurs_solutions():
        Images_Sol = []
        for sol in recuit_simule.Historique_Sol:
            val = recuit_simule.Fonction_objectif(sol)
            Images_Sol.append(val)
        Valeur_Max = max(Images_Sol)
        meilleurs_solutions = []
        for sol, val in zip(recuit_simule.Historique_Sol, Images_Sol):
            if val == Valeur_Max:
                meilleurs_solutions.append(sol)
        uniques = []
        for arr in meilleurs_solutions:
            deja_present = False
            for u in uniques:
                if np.array_equal(arr, u):
                    deja_present = True
                    break
            if not deja_present:
                uniques.append(arr.copy())
        return uniques

### Exécution 

In [ ]:
Solution_initiale = recuit_simule.Solution_Initiale()                         
recuit_simule.Sauvegarder(Solution_initiale)  # Initialiser le circuit avec une solution Initiale

In [ ]:
debut= time.time()
recuit_simule.Recuit_simulé(110,0.99,0.1,15)
fin= time.time()
temps= fin - debut

In [ ]:
heures = int(temps // 3600)
minutes = int ((temps % 3600) // 60)
secondes =  temps % 60
print(f"Temps d'exécution : {heures} h {minutes} min {secondes} s")

### Solutions réalisables

In [ ]:
recuit_simule.Historique_Sol

### Meilleur(s) solution(s) réalisables

In [ ]:
x=recuit_simule.Meilleurs_solutions()

In [ ]:
x

### Indices des 1 de la solutions

In [ ]:
sol = x[0]
indices = [i+1 for i,v in enumerate(sol) if v == 1]
print(indices)

### Evolution du processus 

In [ ]:
Images_Sol= []
for i in recuit_simule.Historique_Sol :
    Images_Sol.append(recuit_simule.Fonction_objectif(i))
plt.plot(Images_Sol) # Images des solutions par Fonction_objectif 
plt.grid()
plt.show()

### Valeur de la meilleure solution par Fonction_objectif 

In [ ]:
recuit_simule.Fonction_objectif( x[0]) 

### A Exécuter avant d'utiliser une autre instance

In [ ]:
recuit_simule.n=0
recuit_simule.C=0
recuit_simule.P=[]
recuit_simule.k=0
recuit_simule.W=[]

### _______________________________________________________________________________________________________________

### _______________________________________________________________________________________________________________

### _______________________________________________________________________________________________________________

### _______________________________________________________________________________________________________________